In [ ]:
"""Bước 1: tự tính khoảng cách và tìm láng giềng, chưa dùng KNN thư viện."""

from pathlib import Path

import numpy as np
import pandas as pd


ROOT = Path.cwd()

# A và C học gần giống nhau, nhưng C có mức chi tiêu khác A nhiều.
ba_ban = {
    "A": np.array([5.0, 3.0, 4000.0]),
    "B": np.array([25.0, 9.0, 4100.0]),
    "C": np.array([6.0, 3.5, 7000.0]),
}


def kc(u, v):
    """Khoảng cách Euclid giữa hai điểm."""
    return np.sqrt(((u - v) ** 2).sum())


print("Khoảng cách KHI CHƯA chuẩn hóa")
print(f"  d(A, B) = {kc(ba_ban['A'], ba_ban['B']):8.1f}")
print(f"  d(A, C) = {kc(ba_ban['A'], ba_ban['C']):8.1f}")

df = pd.read_csv(ROOT / "data" / "sinh_vien_mo_rong.csv")
X = df[["gio_on", "diem_giua_ky", "chi_tieu"]].to_numpy(dtype=float)
trung_binh = X.mean(axis=0)
do_lech = X.std(axis=0)
print("\nĐộ lệch chuẩn của ba cột:")
print(
    f"  gio_on {do_lech[0]:.2f}  diem_giua_ky {do_lech[1]:.2f}"
    f"  chi_tieu {do_lech[2]:.0f}"
)

chuan = {ten: (gia_tri - trung_binh) / do_lech for ten, gia_tri in ba_ban.items()}
print("\nKhoảng cách SAU KHI chuẩn hóa")
print(f"  d(A, B) = {kc(chuan['A'], chuan['B']):.2f}")
print(f"  d(A, C) = {kc(chuan['A'], chuan['C']):.2f}")

# Tìm 5 láng giềng gần nhất cho một bạn mới bằng hai cột học tập.
X2 = df[["gio_on", "diem_giua_ky"]].to_numpy(dtype=float)
y = df["qua_mon"].to_numpy()
tb2, dl2 = X2.mean(axis=0), X2.std(axis=0)
ban_moi = np.array([12.0, 6.5])
X2_chuan = (X2 - tb2) / dl2
ban_moi_chuan = (ban_moi - tb2) / dl2
khoang_cach = np.sqrt(((X2_chuan - ban_moi_chuan) ** 2).sum(axis=1))
gan_nhat = np.argsort(khoang_cach)[:5]

print("\nBạn mới: ôn 12 giờ, điểm giữa kỳ 6.5")
print("Năm láng giềng gần nhất:")
for i in gan_nhat:
    ket_qua = "qua" if y[i] == 1 else "rớt"
    print(
        f"  giờ {X2[i, 0]:5.1f}  điểm {X2[i, 1]:4.1f}  "
        f"khoảng cách {khoang_cach[i]:.3f} -> {ket_qua}"
    )
so_qua = int(y[gan_nhat].sum())
print(f"Bỏ phiếu: {so_qua} qua, {5 - so_qua} rớt")
print("Dự đoán:", "qua môn" if so_qua >= 3 else "rớt môn")


Khoảng cách KHI CHƯA chuẩn hóa
  d(A, B) =    102.2
  d(A, C) =   3000.0

Độ lệch chuẩn của ba cột:
  gio_on 7.82  diem_giua_ky 2.09  chi_tieu 1692

Khoảng cách SAU KHI chuẩn hóa
  d(A, B) = 3.85
  d(A, C) = 1.79

Bạn mới: ôn 12 giờ, điểm giữa kỳ 6.5
Năm láng giềng gần nhất:
  giờ  10.5  điểm  6.4  khoảng cách 0.198 -> qua
  giờ  12.5  điểm  6.1  khoảng cách 0.202 -> qua
  giờ  14.0  điểm  6.4  khoảng cách 0.260 -> qua
  giờ   9.5  điểm  6.1  khoảng cách 0.373 -> rớt
  giờ  10.5  điểm  5.8  khoảng cách 0.386 -> qua
Bỏ phiếu: 4 qua, 1 rớt
Dự đoán: qua môn


In [ ]:
"""Bước 2: đo hậu quả của việc quên chuẩn hóa trên tập kiểm tra."""

from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "sinh_vien_mo_rong.csv")
y = df["qua_mon"]

X3 = df[["gio_on", "diem_giua_ky", "chi_tieu"]]
X_train, X_test, y_train, y_test = train_test_split(
    X3, y, test_size=0.25, random_state=17, stratify=y
)


def danh_gia(model, columns):
    model.fit(X_train[columns], y_train)
    return accuracy_score(y_test, model.predict(X_test[columns]))


def knn_chuan_hoa():
    return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))


ba_cot = ["gio_on", "diem_giua_ky", "chi_tieu"]
hai_cot = ["gio_on", "diem_giua_ky"]

khong_chuan = danh_gia(KNeighborsClassifier(n_neighbors=7), ba_cot)
co_chuan = danh_gia(knn_chuan_hoa(), ba_cot)
bo_chi_tieu = danh_gia(knn_chuan_hoa(), hai_cot)

print(f"Đoán bừa cả lớp qua              : {y_test.mean():.4f}")
print(f"Ba cột, không chuẩn hóa          : {khong_chuan:.4f}")
print(f"Ba cột, có chuẩn hóa             : {co_chuan:.4f}")
print(f"Bỏ cột chi_tieu, có chuẩn hóa    : {bo_chi_tieu:.4f}")


Đoán bừa cả lớp qua              : 0.6000
Ba cột, không chuẩn hóa          : 0.5333
Ba cột, có chuẩn hóa             : 0.8667
Bỏ cột chi_tieu, có chuẩn hóa    : 0.9000


In [ ]:
"""Bước 3: chọn số láng giềng k bằng kiểm định chéo."""

from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("Accuracy trên tập học và tập kiểm tra theo k")
print(f"{'k':>4} {'tập học':>10} {'tập kiểm tra':>14}")
for k in [1, 5, 13, 31, 101]:
    model = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    print(
        f"{k:4d} {accuracy_score(y_train, model.predict(X_train)):10.4f} "
        f"{accuracy_score(y_test, model.predict(X_test)):14.4f}"
    )

print("\nKiểm định chéo 5 phần trên tập học:")
scores = {
    k: cross_val_score(KNeighborsClassifier(n_neighbors=k), X_train, y_train, cv=5).mean()
    for k in range(1, 52, 2)
}
k_tot = max(scores, key=scores.get)
for k in [1, 5, 9, 13, 17, 25, 51]:
    note = " <- tốt nhất" if k == k_tot else ""
    print(f"k = {k:2d}  điểm trung bình {scores[k]:.4f}{note}")

model = KNeighborsClassifier(n_neighbors=k_tot).fit(X_train, y_train)
print(f"\nk tốt nhất = {k_tot}")
print(f"Accuracy trên tập kiểm tra = {accuracy_score(y_test, model.predict(X_test)):.4f}")


Accuracy trên tập học và tập kiểm tra theo k
   k    tập học   tập kiểm tra
   1     1.0000         0.8000
   5     0.8756         0.9067
  13     0.8844         0.9200
  31     0.8667         0.8933
 101     0.8311         0.8533

Kiểm định chéo 5 phần trên tập học:
k =  1  điểm trung bình 0.7778
k =  5  điểm trung bình 0.8444
k =  9  điểm trung bình 0.8444
k = 13  điểm trung bình 0.8667 <- tốt nhất
k = 17  điểm trung bình 0.8533
k = 25  điểm trung bình 0.8622
k = 51  điểm trung bình 0.8533

k tốt nhất = 13
Accuracy trên tập kiểm tra = 0.9200


In [ ]:
"""Bước 4: so sánh KNN với hai mô hình đã học trên dữ liệu có biên cong."""

from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

models = [
    ("Hồi quy logistic", LogisticRegression()),
    ("Naive Bayes", GaussianNB()),
    ("KNN, k = 13", KNeighborsClassifier(n_neighbors=13)),
]

print("Accuracy trên tập kiểm tra, bộ hai vầng trăng")
for name, model in models:
    model.fit(X_train, y_train)
    print(f"  {name:<18}: {accuracy_score(y_test, model.predict(X_test)):.4f}")

print("\nKNN phù hợp hơn vì có thể tạo biên quyết định cong theo hình dạng dữ liệu.")


Accuracy trên tập kiểm tra, bộ hai vầng trăng
  Hồi quy logistic  : 0.8533
  Naive Bayes       : 0.8533
  KNN, k = 13       : 0.9200

KNN phù hợp hơn vì có thể tạo biên quyết định cong theo hình dạng dữ liệu.


In [ ]:
"""Bước 5: KNN cho bài toán hồi quy trên dữ liệu giá nhà."""

from pathlib import Path

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

knn = KNeighborsRegressor(n_neighbors=5).fit(X_train, y_train)
linear = LinearRegression().fit(X_train, y_train)

print("MAE trên tập kiểm tra (tỷ đồng)")
mae_knn = mean_absolute_error(y_test, knn.predict(X_test))
mae_linear = mean_absolute_error(y_test, linear.predict(X_test))
print(f"  KNN, k = 5 : {mae_knn:.4f}")
print(f"  Đường thẳng : {mae_linear:.4f}")

new_houses = pd.DataFrame({"dien_tich": [60.0, 100.0, 150.0, 250.0]})
print("\nDự đoán cho bốn căn mới (tỷ đồng)")
print(f"{'diện tích':>10} {'KNN':>8} {'đường thẳng':>12}")
for area, knn_price, linear_price in zip(
    new_houses["dien_tich"], knn.predict(new_houses), linear.predict(new_houses)
):
    print(f"{area:7.0f} m2 {knn_price:8.2f} {linear_price:12.2f}")

_, indices = knn.kneighbors(pd.DataFrame({"dien_tich": [250.0]}))
neighbors = X_train.iloc[indices[0]]["dien_tich"].tolist()
print("\nNăm láng giềng của căn 250 m2:", ", ".join(f"{value:.1f}" for value in neighbors))
print("KNN không ngoại suy: ngoài vùng dữ liệu, nó vẫn dùng các căn lớn nhất đã thấy.")


MAE trên tập kiểm tra (tỷ đồng)
  KNN, k = 5 : 0.4030
  Đường thẳng : 0.2578

Dự đoán cho bốn căn mới (tỷ đồng)
 diện tích      KNN  đường thẳng
     60 m2     4.95         5.12
    100 m2     8.32         8.24
    150 m2     9.32        12.13
    250 m2     9.32        19.91

Năm láng giềng của căn 250 m2: 117.5, 115.9, 111.7, 111.2, 104.2
KNN không ngoại suy: ngoài vùng dữ liệu, nó vẫn dùng các căn lớn nhất đã thấy.
